# Avance 3. Baseline

## Equipo 41

**Proyecto:** Detección de ocultamiento de mercancía en video para retail

### Objetivo

Este notebook documenta el desarrollo y evaluación del modelo baseline utilizado como referencia para los experimentos posteriores del proyecto integrador. El objetivo es establecer un punto de comparación reproducible que permita evaluar la viabilidad de distinguir eventos de `ocultamiento` y `sin_ocultamiento` a partir de clips de video.

Debido a que el problema corresponde al reconocimiento de acciones en video, se utiliza una arquitectura R3D-18 implementada en PyTorch y preentrenada sobre Kinetics-400. Esta arquitectura permite procesar directamente secuencias RGB y aprender representaciones espacio-temporales a partir de la información visual y del movimiento presente entre frames. El modelo se adapta al problema mediante transferencia de aprendizaje y ajuste fino para clasificación binaria.

El desarrollo del baseline sigue la fase de modelado de CRISP-ML(Q), considerando tanto el desempeño predictivo como aspectos relacionados con generalización, complejidad, explicabilidad y demanda de recursos. La evaluación se realiza sobre conjuntos separados de entrenamiento, validación y prueba, manteniendo controles para evitar fuga de información entre clips relacionados y conservando la fuente como variable de control para analizar generalización.

Además del desempeño del modelo, este notebook analiza las representaciones aprendidas por R3D-18, el posible subajuste o sobreajuste durante el entrenamiento, las métricas apropiadas para el problema de clasificación binaria y el nivel de desempeño alcanzado respecto a referencias básicas.

### Relación entre la rúbrica y el tratamiento del baseline de video

La rúbrica del Avance 3 está formulada de manera general para problemas de aprendizaje automático. En este proyecto, sus criterios se adaptan a un problema de clasificación binaria de acciones sobre datos de video no estructurados. Esta adaptación conserva el propósito de cada criterio, pero considera que las características utilizadas para la clasificación no corresponden a variables tabulares definidas manualmente, sino a representaciones espacio-temporales aprendidas por una red neuronal convolucional 3D.

| Criterio de la rúbrica | Aplicación en este notebook |
|---|---|
| Algoritmo | Selección y justificación de R3D-18 como modelo baseline para reconocimiento de acciones en video. Se consideran la naturaleza espacio-temporal de los datos, el tamaño disponible del conjunto de entrenamiento, el uso de pesos preentrenados en Kinetics-400, la complejidad del modelo, sus requerimientos computacionales y sus limitaciones de interpretabilidad. |
| Características importantes | Las características no se seleccionan directamente a partir de columnas tabulares. R3D-18 aprende representaciones espacio-temporales durante el procesamiento de los clips. Se extraen los embeddings generados antes de la capa de clasificación y se analiza su estructura mediante reducción de dimensionalidad y representación visual. |
| Subajuste / sobreajuste | Se compara la evolución de la función de pérdida y de las métricas de desempeño entre entrenamiento y validación. También se comparan los resultados obtenidos en entrenamiento, validación y prueba para identificar posibles brechas de generalización. |
| Métrica | La evaluación se adapta a un problema de clasificación binaria con desbalance de clases. Las métricas principales serán ROC-AUC, PR-AUC y EER, dando especial atención a PR-AUC por el comportamiento sobre la clase minoritaria. Se complementan con matriz de confusión, precision, recall, F1-score, specificity y accuracy para analizar el comportamiento a un umbral de decisión determinado. |
| Desempeño | El resultado del modelo se compara contra referencias básicas y contra el comportamiento esperado de un clasificador no informativo. El desempeño obtenido se utiliza como punto de referencia para determinar posteriormente si modelos alternativos proporcionan mejoras relevantes. |

La selección y evaluación del modelo se realiza siguiendo CRISP-ML(Q), por lo que el análisis no se limita a una única métrica predictiva. También se documentan aspectos relacionados con generalización, tiempo de entrenamiento, recursos utilizados, complejidad y capacidad de interpretación de las representaciones aprendidas.

## 1. Definición del problema de modelado

El proyecto se formula como un problema de aprendizaje supervisado para clasificación binaria de acciones en video. El objetivo del modelo es distinguir entre clips que contienen un evento observable de `ocultamiento` de mercancía y clips que representan comportamiento `sin_ocultamiento`.

La unidad de análisis es un clip temporal de video y no el video completo. Esta decisión es necesaria porque un video puede contener diferentes acciones a lo largo del tiempo y un evento de ocultamiento puede representar solamente una fracción de su duración. En consecuencia, la etiqueta asignada a un video no se transfiere automáticamente a todos sus fragmentos.

Para el baseline, cada observación se representa mediante una secuencia RGB de 16 frames con resolución espacial de 112 × 112 píxeles. La entrada esperada por el modelo tiene la forma:

$$
X_i \in \mathbb{R}^{3 \times 16 \times 112 \times 112}
$$

La variable objetivo se define como:

$$
y_i =
\begin{cases}
0, & \text{sin\_ocultamiento} \\
1, & \text{ocultamiento}
\end{cases}
$$

La clase `ocultamiento` corresponde a clips en los que existe evidencia visual suficiente de acercamiento o introducción de mercancía hacia una zona definida de ocultamiento, como bolsillo, ropa, bolsa o mochila. La clase `sin_ocultamiento` incluye comportamientos observables donde la mercancía permanece visible o no se presenta la acción objetivo.

Los clips cuya acción no pueda determinarse con suficiente claridad debido a oclusiones, visibilidad insuficiente, cortes temporales u otras condiciones ambiguas no se fuerzan a una de las dos clases para el entrenamiento principal. Estos casos se conservan, cuando resulte pertinente, para análisis posterior de errores y limitaciones.

El modelo genera dos valores de salida asociados con las clases del problema. Durante el entrenamiento estos valores se utilizan como logits para calcular la función de pérdida y, durante la evaluación, pueden transformarse mediante Softmax en probabilidades estimadas para cada clase.

### 1.1 Decisiones de modelado y criterios de calidad

La definición del baseline considera las características de los datos, las restricciones del proyecto y el propósito de utilizar el modelo como referencia para experimentos posteriores. De acuerdo con CRISP-ML(Q), la selección y evaluación del modelo debe considerar tanto el desempeño predictivo como su capacidad de generalización y las restricciones relevantes para su posible utilización.

| Elemento | Decisión para el baseline |
|---|---|
| Tipo de aprendizaje | Supervisado |
| Tipo de problema | Clasificación binaria de acciones en video |
| Clase positiva | `ocultamiento` |
| Clase negativa | `sin_ocultamiento` |
| Unidad de análisis | Clip temporal |
| Entrada | Video RGB |
| Frames por muestra | 16 |
| Resolución de entrada | 112 × 112 |
| Modelo baseline | R3D-18 |
| Framework | PyTorch / Torchvision |
| Inicialización | Pesos preentrenados en Kinetics-400 |
| Estrategia de entrenamiento | Transfer learning y ajuste fino |
| Salida durante entrenamiento | Dos logits, uno por clase |
| Salida para evaluación | Probabilidades obtenidas mediante Softmax |
| Separación de datos | Entrenamiento, validación y prueba sin compartir entre particiones clips pertenecientes al mismo grupo (dataset, video_id) |
| Tratamiento de casos ambiguos | Exclusión del entrenamiento principal y conservación para análisis cuando sea pertinente |
| Uso previsto | Referencia experimental para comparar modelos posteriores |
| Alcance | Evaluación sobre videos grabados; no corresponde a un sistema productivo |

Además del desempeño predictivo, se consideran los siguientes criterios de calidad:

- **Generalización:** diferencia entre el desempeño obtenido en entrenamiento, validación y prueba.
- **Explicabilidad:** análisis posterior de las representaciones espacio-temporales aprendidas y de los errores de clasificación.
- **Complejidad:** número de parámetros y características de la arquitectura.
- **Demanda de recursos:** tiempo de entrenamiento, memoria utilizada y velocidad de inferencia.
- **Reproducibilidad:** registro de configuración, semillas, particiones, hiperparámetros y versiones de las bibliotecas utilizadas.

La clasificación producida por el modelo se mantiene separada de la lógica operativa de generación de alertas. El baseline estima la probabilidad de cada clase para un clip; la selección posterior de un umbral de decisión permitirá analizar el compromiso entre detecciones correctas y falsas alarmas.

## 2. Protocolo experimental y partición de datos

El Avance 2 estableció las condiciones que deben conservarse durante el
modelado para evitar fuga de información y controlar los principales riesgos
identificados durante Data Preparation.

En particular:

- Únicamente participan los clips elegibles para clasificación binaria;
- Los clips pertenecientes al mismo grupo `(dataset, video_id)` deben
  permanecer dentro de una sola partición;
- La composición de las particiones debe conservar, en la medida de lo
  posible, representación de ambas clases y de las diferentes fuentes;
- Cualquier tratamiento del desbalance se realizará después de la
  partición y exclusivamente sobre entrenamiento;
- La asignación a entrenamiento, validación y prueba se persistirá para
  utilizar exactamente las mismas observaciones al comparar arquitecturas.

A partir de estas condiciones, en este avance se construye y valida la
partición experimental que utilizará el baseline R3D-18.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Directorio de trabajo para la preparación de datos
BASE_DIR = Path(
    "/opt/workspace/MNA---Proyecto-Integrador/"
    "Datos/Preparacion_Datasets"
)

# Archivos generados durante la preparación
MANIFEST_PATH = BASE_DIR / "dataset_manifest_todos.csv"

FEATURES_PATH = (
    BASE_DIR / "dataset_consolidado_features.csv"
)

DURACIONES_PATH = (
    BASE_DIR / "duraciones_dataset_consolidado.csv"
)

# Dataset físico consolidado
CLIPS_DIR = Path(
    "/opt/workspace/datasets/procesados/clips_4s"
)

print("Manifest disponible:    ", MANIFEST_PATH.exists())
print("Features disponibles:    ", FEATURES_PATH.exists())
print("Duraciones disponibles:  ", DURACIONES_PATH.exists())
print("Directorio de clips:     ", CLIPS_DIR.exists())

# El archivo de features será la tabla principal para definir
# el protocolo experimental del baseline
if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo esperado: {FEATURES_PATH}"
    )

df = pd.read_csv(FEATURES_PATH)

print(f"\nRegistros: {len(df):,}")
print(f"Columnas:  {len(df.columns)}")

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nPrimeros registros:")
display(df.head())

Manifest disponible:     True
Features disponibles:     True
Duraciones disponibles:   True
Directorio de clips:      True

Registros: 987
Columnas:  19

Columnas disponibles:
['dataset', 'video_id', 'video_origen', 'clip_id', 'inicio_s', 'fin_s', 'etiqueta', 'persona_id', 'evento_id', 'nota', 'nombre_clip', 'duracion_real_s', 'duracion_objetivo_s', 'diferencia_vs_4s', 'cumple_4s', 'requiere_normalizacion_temporal', 'elegible_modelo_binario', 'motivo_no_elegible', 'clase_binaria']

Primeros registros:


,dataset,video_id,video_origen,clip_id,inicio_s,fin_s,etiqueta,persona_id,evento_id,nota,nombre_clip,duracion_real_s,duracion_objetivo_s,diferencia_vs_4s,cumple_4s,requiere_normalizacion_temporal,elegible_modelo_binario,motivo_no_elegible,clase_binaria
0,CCTV_YOLO,CCTV_001,not_shoplifting1.mp4,CCTV_001,2.0,6.0,sin_ocultamiento,P001_AJ,NaN,NaN,CCTV_YOLO__CCTV_001__CCTV_001.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
1,CCTV_YOLO,CCTV_002,not_shoplifting2.mp4,CCTV_002,2.0,6.0,sin_ocultamiento,P002_AJ,NaN,NaN,CCTV_YOLO__CCTV_002__CCTV_002.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
2,CCTV_YOLO,CCTV_003,not_shoplifting3.mp4,CCTV_003,2.0,6.0,sin_ocultamiento,P003_AJ,NaN,NaN,CCTV_YOLO__CCTV_003__CCTV_003.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
3,CCTV_YOLO,CCTV_004,not_shoplifting4.mp4,CCTV_004,2.0,6.0,sin_ocultamiento,P004_AJ,NaN,NaN,CCTV_YOLO__CCTV_004__CCTV_004.mp4,4.0,4.0,0.0,True,False,True,NaN,0.0
4,CCTV_YOLO,CCTV_005,shoplifting1.mp4,CCTV_005,2.0,6.0,ocultamiento,P001_AJ,NaN,NaN,CCTV_YOLO__CCTV_005__CCTV_005.mp4,4.0,4.0,0.0,True,False,True,NaN,1.0


### 2.1 Unidad de agrupación

El Avance 2 definió la combinación `(dataset, video_id)` como unidad mínima
de agrupación para prevenir fuga de información entre clips relacionados.
Esta regla se conserva durante la construcción de las particiones.

Antes de generar el split definitivo se verifica adicionalmente `persona_id`,
ya que el Avance 2 estableció que los identificadores confiables de persona,
evento o escenario deben considerarse cuando permitan detectar dependencia
entre observaciones pertenecientes a videos distintos.

Para el modelado se utilizan únicamente registros con
`elegible_modelo_binario = True` y una clase binaria definida.

In [2]:
# Subconjunto utilizado para el baseline
df_modelo = df[
    df["elegible_modelo_binario"].eq(True)
    & df["clase_binaria"].notna()
].copy()

df_modelo["clase_binaria"] = (
    df_modelo["clase_binaria"].astype(int)
)

df_modelo["grupo_video"] = (
    df_modelo["dataset"].astype(str)
    + "__"
    + df_modelo["video_id"].astype(str)
)

print(f"Clips elegibles: {len(df_modelo):,}")
print(f"Grupos de video: {df_modelo['grupo_video'].nunique():,}")

assert len(df_modelo) == 985
assert set(df_modelo["clase_binaria"].unique()) == {0, 1}

Clips elegibles: 985
Grupos de video: 229


In [3]:
# Verificar cuántas clases aparecen dentro de cada video
clases_por_video = (
    df_modelo
    .groupby("grupo_video")["clase_binaria"]
    .nunique()
)

videos_multiclase = clases_por_video[
    clases_por_video > 1
]

print(
    "Videos que contienen clips de más de una clase:",
    len(videos_multiclase)
)

Videos que contienen clips de más de una clase: 16


### 2.2 Auditoría de identificadores de persona

El Avance 2 estableció que `(dataset, video_id)` constituye la unidad mínima
de agrupación para evitar fuga de información entre clips relacionados.
También se determinó que identificadores confiables de persona, evento o
escenario deben considerarse cuando permitan detectar dependencia entre
videos diferentes.

Antes de construir las particiones se revisa `persona_id` para determinar:

1. Qué proporción de los clips dispone de un identificador de persona;
2. Si los identificadores son consistentes dentro de cada fuente;
3. Si una misma persona aparece asociada con más de un grupo de video;
4. Si esas relaciones requieren ampliar la unidad de agrupación utilizada
   para separar entrenamiento, validación y prueba.

Los identificadores se interpretan únicamente dentro de cada dataset. Dos
valores iguales en fuentes diferentes no se consideran automáticamente la
misma persona, ya que cada fuente utiliza su propio esquema de identificación.
Los valores que representan identidad desconocida, como `PNA`, se excluyen
de este análisis.

In [4]:
# ------------------------------------------------------------
# Auditoría de persona_id antes de construir las particiones
# ------------------------------------------------------------

PERSONAS_DESCONOCIDAS = {
    "",
    "PNA",
    "P0NA",
    "NA",
    "N/A",
    "NONE",
    "UNKNOWN",
    "DESCONOCIDO"
}

df_personas = df_modelo.copy()

# Normalizar identificadores
df_personas["persona_id_limpio"] = (
    df_personas["persona_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Identificar registros con persona conocida
df_personas["persona_conocida"] = (
    ~df_personas["persona_id_limpio"]
    .str.upper()
    .isin(PERSONAS_DESCONOCIDAS)
)

print("Disponibilidad de persona_id por dataset:")
display(
    pd.crosstab(
        df_personas["dataset"],
        df_personas["persona_conocida"],
        margins=True
    ).rename(
        columns={
            False: "desconocida",
            True: "conocida"
        }
    )
)

# Trabajar únicamente con IDs conocidos
df_personas_conocidas = (
    df_personas[
        df_personas["persona_conocida"]
    ]
    .copy()
)

# El identificador se interpreta dentro de cada fuente
df_personas_conocidas["grupo_persona"] = (
    df_personas_conocidas["dataset"].astype(str)
    + "__"
    + df_personas_conocidas["persona_id_limpio"]
)

# Número de grupos de video asociados con cada persona
resumen_personas = (
    df_personas_conocidas
    .groupby(
        [
            "dataset",
            "grupo_persona",
            "persona_id_limpio"
        ],
        as_index=False
    )
    .agg(
        n_grupos_video=("grupo_video", "nunique"),
        n_clips=("clip_id", "size"),
        n_clases=("clase_binaria", "nunique")
    )
)

personas_multivideo = (
    resumen_personas[
        resumen_personas["n_grupos_video"] > 1
    ]
    .sort_values(
        ["n_grupos_video", "n_clips"],
        ascending=False
    )
)

print(
    "\nPersonas conocidas:",
    resumen_personas["grupo_persona"].nunique()
)

print(
    "Personas presentes en más de un grupo de video:",
    len(personas_multivideo)
)

if len(personas_multivideo) > 0:
    display(personas_multivideo)

    print("\nVideos asociados con personas multivideo:")

    detalle_multivideo = (
        df_personas_conocidas[
            df_personas_conocidas["grupo_persona"].isin(
                personas_multivideo["grupo_persona"]
            )
        ][
            [
                "dataset",
                "persona_id_limpio",
                "grupo_video",
                "video_origen",
                "clase_binaria"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "dataset",
                "persona_id_limpio",
                "grupo_video",
                "clase_binaria"
            ]
        )
    )

    display(detalle_multivideo)

Disponibilidad de persona_id por dataset:


persona_conocida,desconocida,conocida,All
dataset,,,
CCTV_YOLO,0,8,8
DCSASS,419,364,783
MNNIT,6,176,182
SINTETICO,0,12,12
All,425,560,985



Personas conocidas: 51
Personas presentes en más de un grupo de video: 16


,dataset,grupo_persona,persona_id_limpio,n_grupos_video,n_clips,n_clases
25,MNNIT,MNNIT__P001,P001,36,36,2
31,MNNIT,MNNIT__P007,P007,31,31,2
30,MNNIT,MNNIT__P006,P006,28,28,2
32,MNNIT,MNNIT__P008,P008,17,17,2
34,MNNIT,MNNIT__P010,P010,13,13,2
26,MNNIT,MNNIT__P002,P002,10,10,2
33,MNNIT,MNNIT__P009,P009,10,10,2
27,MNNIT,MNNIT__P003,P003,8,8,2
28,MNNIT,MNNIT__P004,P004,8,8,2
29,MNNIT,MNNIT__P005,P005,6,6,2



Videos asociados con personas multivideo:


,dataset,persona_id_limpio,grupo_video,video_origen,clase_binaria
0,CCTV_YOLO,P001_AJ,CCTV_YOLO__CCTV_001,not_shoplifting1.mp4,0
4,CCTV_YOLO,P001_AJ,CCTV_YOLO__CCTV_005,shoplifting1.mp4,1
1,CCTV_YOLO,P002_AJ,CCTV_YOLO__CCTV_002,not_shoplifting2.mp4,0
5,CCTV_YOLO,P002_AJ,CCTV_YOLO__CCTV_006,shoplifting2.mp4,1
2,CCTV_YOLO,P003_AJ,CCTV_YOLO__CCTV_003,not_shoplifting3.mp4,0
...,...,...,...,...,...
95,MNNIT,P012,MNNIT__shoplifting-89,shoplifting-89.mp4,1
96,MNNIT,P012,MNNIT__shoplifting-90,shoplifting-90.mp4,1
189,MNNIT,P014,MNNIT__normal-90,normal-90.mp4,0
98,MNNIT,P014,MNNIT__shoplifting-92,shoplifting-92.mp4,1


### 2.2.1 Resultado de la auditoría de persona

La auditoría confirmó que `persona_id` aporta información adicional sobre la
dependencia entre observaciones.

En particular, se identificaron personas conocidas que aparecen en varios
`video_id` dentro de una misma fuente. Este comportamiento es especialmente
relevante en MNNIT, donde algunos sujetos participan en múltiples videos de
ambas clases, y en CCTV_YOLO, donde una misma persona aparece en ejemplos de
`ocultamiento` y `sin_ocultamiento`.

Por esta razón, utilizar únicamente `(dataset, video_id)` como criterio de
separación podría colocar videos de una misma persona en entrenamiento y
evaluación. Esto mantendría separados los archivos de video, pero no impediría
que el modelo encontrara nuevamente al mismo sujeto.

La unidad definitiva de agrupación se construirá manteniendo juntos todos los
videos relacionados mediante identificadores de persona conocidos y
confiables. Los videos sin un identificador de persona disponible conservarán
como grupo mínimo la combinación `(dataset, video_id)`.

Los identificadores utilizados como marcadores de persona desconocida, como
`PNA` y `P0NA`, no se emplean para relacionar videos entre sí.

In [5]:
# ------------------------------------------------------------
# Construcción de grupos experimentales
# ------------------------------------------------------------

PERSONAS_DESCONOCIDAS = {
    "",
    "PNA",
    "P0NA",
    "NA",
    "N/A",
    "NONE",
    "UNKNOWN",
    "DESCONOCIDO"
}

df_split = df_modelo.copy()

df_split["persona_id_limpio"] = (
    df_split["persona_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_split["persona_conocida"] = (
    ~df_split["persona_id_limpio"]
    .str.upper()
    .isin(PERSONAS_DESCONOCIDAS)
)


# ------------------------------------------------------------
# Union-Find para conectar videos relacionados por persona
# ------------------------------------------------------------

parent = {}


def find(x):
    if x not in parent:
        parent[x] = x

    if parent[x] != x:
        parent[x] = find(parent[x])

    return parent[x]


def union(a, b):
    raiz_a = find(a)
    raiz_b = find(b)

    if raiz_a != raiz_b:
        parent[raiz_b] = raiz_a


# Cada video comienza como un grupo independiente
for grupo_video in df_split["grupo_video"].unique():
    find(grupo_video)


# Dentro de cada dataset, unir videos que comparten
# una persona conocida
df_con_id = df_split[
    df_split["persona_conocida"]
].copy()

for (dataset, persona_id), grupo in df_con_id.groupby(
    ["dataset", "persona_id_limpio"]
):
    videos = grupo["grupo_video"].unique()

    if len(videos) > 1:
        primero = videos[0]

        for video in videos[1:]:
            union(primero, video)


# Aplicar compresión de caminos y generar IDs reproducibles
df_split["raiz_grupo"] = (
    df_split["grupo_video"]
    .map(find)
)

raices = sorted(
    df_split["raiz_grupo"].unique()
)

mapa_grupos = {
    raiz: f"G{idx:03d}"
    for idx, raiz in enumerate(raices, start=1)
}

df_split["grupo_experimental"] = (
    df_split["raiz_grupo"]
    .map(mapa_grupos)
)


# ------------------------------------------------------------
# Resumen de los grupos resultantes
# ------------------------------------------------------------

resumen_grupos = (
    df_split
    .groupby("grupo_experimental")
    .agg(
        n_clips=("clip_id", "size"),
        n_videos=("grupo_video", "nunique"),
        n_personas_conocidas=(
            "persona_id_limpio",
            lambda x: x[
                ~x.str.upper().isin(PERSONAS_DESCONOCIDAS)
            ].nunique()
        ),
        n_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 1).sum()
        ),
        n_sin_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

print(f"Clips:                  {len(df_split):,}")
print(
    f"Videos originales:      "
    f"{df_split['grupo_video'].nunique():,}"
)
print(
    f"Grupos experimentales:  "
    f"{df_split['grupo_experimental'].nunique():,}"
)

print("\nTamaño de los grupos experimentales:")
display(
    resumen_grupos[
        ["n_clips", "n_videos"]
    ]
    .describe()
)

print("\nGrupos con más videos relacionados:")
display(
    resumen_grupos
    .sort_values(
        ["n_videos", "n_clips"],
        ascending=False
    )
    .head(15)
)

Clips:                  985
Videos originales:      229
Grupos experimentales:  63

Tamaño de los grupos experimentales:


,n_clips,n_videos
count,63.000000,63.000000
mean,15.634921,3.634921
std,13.993433,7.092253
min,1.000000,1.000000
25%,1.000000,1.000000
50%,13.000000,1.000000
75%,32.000000,1.500000
max,36.000000,36.000000



Grupos con más videos relacionados:


,grupo_experimental,n_clips,n_videos,n_personas_conocidas,n_ocultamiento,n_sin_ocultamiento
37,G038,36,36,1,17,19
38,G039,31,31,1,15,16
47,G048,28,28,1,12,16
39,G040,17,17,1,11,6
43,G044,13,13,1,4,9
40,G041,10,10,1,6,4
41,G042,10,10,1,5,5
42,G043,8,8,1,4,4
44,G045,8,8,1,6,2
46,G047,6,6,1,5,1


### 2.3 Construcción de las particiones experimentales

Antes de realizar la separación en entrenamiento, validación y prueba se
construyó una unidad denominada `grupo_experimental`. Su propósito es evitar
que observaciones estrechamente relacionadas aparezcan en particiones
diferentes y produzcan fuga de información.

Un grupo experimental reúne todos los clips que deben mantenerse juntos
durante la partición. Como condición mínima, todos los clips pertenecientes
al mismo `(dataset, video_id)` forman parte del mismo grupo. Adicionalmente,
cuando existe un `persona_id` conocido y confiable, se agrupan también los
videos de una misma fuente en los que aparece esa persona.

La relación se aplica de forma transitiva. Por ejemplo, si una persona aparece
en los videos A y B, y otra persona relaciona los videos B y C, los videos
A, B y C forman un único grupo experimental. De esta manera se evita que
videos relacionados indirectamente puedan quedar separados entre
entrenamiento y evaluación.

Los identificadores que representan una persona desconocida, como `PNA` o
`P0NA`, no se utilizan para unir videos. En esos casos se mantiene únicamente
la agrupación mínima definida por `(dataset, video_id)`.

Después de aplicar estas reglas, los 985 clips elegibles y 229 videos originales
quedaron organizados en 63 grupos experimentales indivisibles. Estos grupos,
y no los clips individuales, constituyen las unidades que se asignarán a las
particiones.

Esta estrategia es necesaria porque una partición aleatoria a nivel de clip,
o incluso únicamente a nivel de archivo de video, podría permitir que el
modelo observara durante el entrenamiento escenas o personas estrechamente
relacionadas con las utilizadas posteriormente para validación o prueba. En
ese caso, las métricas podrían reflejar parcialmente reconocimiento de la
persona, apariencia, escenario u otras características compartidas, en lugar
de generalización de la acción de ocultamiento.

Se adopta como objetivo una distribución aproximada de:

- 70 % para entrenamiento;
- 15 % para validación;
- 15 % para prueba.

Estas proporciones son objetivos y no restricciones exactas. La prioridad es
preservar la independencia entre los 63 grupos experimentales. Dentro de esta
restricción se busca que las particiones mantengan una composición
razonablemente similar respecto al número de clips, las dos clases y las
fuentes de datos.

La partición se obtiene mediante una búsqueda reproducible entre múltiples
asignaciones posibles de los grupos experimentales. Cada solución candidata
se evalúa de acuerdo con su desviación respecto a las proporciones objetivo
en número de clips, distribución de `ocultamiento` y `sin_ocultamiento`, y
representación de las fuentes.

El conjunto de prueba permanecerá aislado durante el desarrollo del modelo.
La selección de checkpoints, hiperparámetros y umbrales de decisión se
realizará utilizando únicamente entrenamiento y validación.

La asignación final se conservará como un artefacto reproducible para que
R3D-18 y las arquitecturas evaluadas posteriormente utilicen exactamente las
mismas particiones.

In [6]:
# ------------------------------------------------------------
# Resumen de información necesaria para el split
# ------------------------------------------------------------

resumen_split = (
    df_split
    .groupby("grupo_experimental")
    .agg(
        dataset=("dataset", "first"),
        n_clips=("clip_id", "size"),
        n_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 1).sum()
        ),
        n_sin_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 0).sum()
        ),
        n_videos=("grupo_video", "nunique")
    )
    .reset_index()
)

# Verificar que ningún grupo experimental mezcle fuentes
datasets_por_grupo = (
    df_split
    .groupby("grupo_experimental")["dataset"]
    .nunique()
)

assert datasets_por_grupo.max() == 1

print("Grupos experimentales por fuente:")
display(
    resumen_split["dataset"]
    .value_counts()
    .rename_axis("dataset")
    .to_frame("grupos")
)

print("\nClips contenidos en esos grupos:")
display(
    resumen_split
    .groupby("dataset")["n_clips"]
    .sum()
    .to_frame("clips")
)

Grupos experimentales por fuente:


,grupos
dataset,
DCSASS,27
MNNIT,20
SINTETICO,12
CCTV_YOLO,4



Clips contenidos en esos grupos:


,clips
dataset,
CCTV_YOLO,8
DCSASS,783
MNNIT,182
SINTETICO,12


In [ ]:
import numpy as np

# ------------------------------------------------------------
# Configuración de la partición
# ------------------------------------------------------------

SPLITS = ("train", "val", "test")

PROPORCIONES = {
    "train": 0.70,
    "val": 0.15,
    "test": 0.15
}

SEED = 41
N_INTENTOS = 100000

rng = np.random.default_rng(SEED)


# ------------------------------------------------------------
# Totales globales y objetivos
# ------------------------------------------------------------

total_clips = resumen_split["n_clips"].sum()
total_pos = resumen_split["n_ocultamiento"].sum()
total_neg = resumen_split["n_sin_ocultamiento"].sum()

totales_dataset = (
    resumen_split
    .groupby("dataset")["n_clips"]
    .sum()
    .to_dict()
)


def evaluar_asignacion(asignacion):
    """
    Menor score = partición más cercana a las proporciones
    objetivo en tamaño, clase y fuente.
    """

    temp = resumen_split.copy()
    temp["split"] = asignacion

    # ----------------------------------------------------
    # Restricciones mínimas de representación por fuente
    # ----------------------------------------------------

    # CCTV_YOLO tiene cuatro grupos experimentales y puede
    # estar representado en las tres particiones.
    for split in SPLITS:
        tiene_cctv = (
            (temp["split"] == split)
            & (temp["dataset"] == "CCTV_YOLO")
        ).any()

        if not tiene_cctv:
            return np.inf


    # El entrenamiento debe contener ejemplos sintéticos
    # de ambas clases para evitar una asociación directa
    # entre fuente sintética y clase positiva.
    sintetico_train = temp[
        (temp["split"] == "train")
        & (temp["dataset"] == "SINTETICO")
    ]

    if sintetico_train["n_ocultamiento"].sum() == 0:
        return np.inf

    if sintetico_train["n_sin_ocultamiento"].sum() == 0:
        return np.inf

    score = 0.0

    for split in SPLITS:
        prop = PROPORCIONES[split]

        subset = temp[temp["split"] == split]

        # Penalización fuerte si algún split queda vacío
        if len(subset) == 0:
            return np.inf

        # ----------------------------------------------------
        # 1. Tamaño total
        # ----------------------------------------------------

        observado = subset["n_clips"].sum()
        objetivo = total_clips * prop

        score += (
            (observado - objetivo) / objetivo
        ) ** 2

        # ----------------------------------------------------
        # 2. Balance de clases
        # ----------------------------------------------------

        observado_pos = subset["n_ocultamiento"].sum()
        objetivo_pos = total_pos * prop

        observado_neg = subset["n_sin_ocultamiento"].sum()
        objetivo_neg = total_neg * prop

        score += 2.0 * (
            (observado_pos - objetivo_pos)
            / objetivo_pos
        ) ** 2

        score += (
            (observado_neg - objetivo_neg)
            / objetivo_neg
        ) ** 2

        # ----------------------------------------------------
        # 3. Representación por fuente
        # ----------------------------------------------------

        clips_por_dataset = (
            subset
            .groupby("dataset")["n_clips"]
            .sum()
            .to_dict()
        )

        for dataset, total_dataset in totales_dataset.items():

            observado_ds = clips_por_dataset.get(dataset, 0)
            objetivo_ds = total_dataset * prop

            # Se normaliza respecto al tamaño total de la fuente.
            # Esto evita que las fuentes pequeñas dominen el score.
            score += 0.5 * (
                (observado_ds - objetivo_ds)
                / total_dataset
            ) ** 2

    return score


# ------------------------------------------------------------
# Búsqueda reproducible
# ------------------------------------------------------------

mejor_score = np.inf
mejor_asignacion = None

probabilidades = np.array(
    [PROPORCIONES[s] for s in SPLITS]
)

for _ in range(N_INTENTOS):

    asignacion = rng.choice(
        SPLITS,
        size=len(resumen_split),
        p=probabilidades
    )

    score = evaluar_asignacion(asignacion)

    if score < mejor_score:
        mejor_score = score
        mejor_asignacion = asignacion.copy()


resumen_split["split"] = mejor_asignacion

print(f"Mejor score encontrado: {mejor_score:.6f}")

display(
    resumen_split[
        [
            "grupo_experimental",
            "dataset",
            "n_clips",
            "n_ocultamiento",
            "n_sin_ocultamiento",
            "n_videos",
            "split"
        ]
    ]
    .sort_values(
        ["split", "dataset", "grupo_experimental"]
    )
)

### 2.4 Validación de las particiones

La función utilizada para construir las particiones selecciona una asignación
de grupos experimentales que aproxima las proporciones objetivo. Sin embargo,
el valor de la función de optimización no constituye por sí mismo evidencia
suficiente de que la partición sea adecuada.

Antes de utilizarla para entrenar el baseline se verifican explícitamente las
siguientes condiciones:

1. Ningún grupo experimental aparece en más de una partición
2. Ninguna persona conocida aparece en más de una partición
3. Entrenamiento, validación y prueba contienen ambas clases
4. Las proporciones de clase se mantienen razonablemente cercanas a la
   distribución global
5. Las diferentes fuentes están representadas en las particiones cuando la
   estructura de los grupos lo permite
6. Las proporciones finales de clips son suficientemente cercanas al objetivo
   de 70/15/15 sin romper las restricciones de independencia.

Estas comprobaciones tienen prioridad sobre obtener porcentajes exactamente
iguales a los objetivos, ya que una división numéricamente más precisa pero
con observaciones relacionadas en diferentes particiones produciría una
estimación optimista del desempeño.

In [ ]:
# ------------------------------------------------------------
# Incorporar el split seleccionado a cada clip
# ------------------------------------------------------------

mapa_split = resumen_split.set_index(
    "grupo_experimental"
)["split"]

df_split["split"] = (
    df_split["grupo_experimental"]
    .map(mapa_split)
)

assert df_split["split"].notna().all()

print(
    df_split["split"]
    .value_counts()
)

split
train    686
test     150
val      149
Name: count, dtype: int64


In [ ]:
# ------------------------------------------------------------
# Resumen general de las particiones
# ------------------------------------------------------------

resumen_particiones = (
    df_split
    .groupby("split")
    .agg(
        clips=("clip_id", "size"),
        grupos=("grupo_experimental", "nunique"),
        videos=("grupo_video", "nunique"),
        ocultamiento=(
            "clase_binaria",
            lambda x: (x == 1).sum()
        ),
        sin_ocultamiento=(
            "clase_binaria",
            lambda x: (x == 0).sum()
        )
    )
    .reindex(["train", "val", "test"])
)

resumen_particiones["porcentaje_clips"] = (
    resumen_particiones["clips"]
    / len(df_split)
    * 100
)

resumen_particiones["pct_ocultamiento"] = (
    resumen_particiones["ocultamiento"]
    / resumen_particiones["clips"]
    * 100
)

resumen_particiones["pct_sin_ocultamiento"] = (
    resumen_particiones["sin_ocultamiento"]
    / resumen_particiones["clips"]
    * 100
)

display(
    resumen_particiones.round(2)
)

,clips,grupos,videos,ocultamiento,sin_ocultamiento,porcentaje_clips,pct_ocultamiento,pct_sin_ocultamiento
split,,,,,,,,
train,686,45,160,135,551,69.64,19.68,80.32
val,149,9,26,29,120,15.13,19.46,80.54
test,150,9,43,26,124,15.23,17.33,82.67


In [ ]:
# ------------------------------------------------------------
# Representación de las fuentes
# ------------------------------------------------------------

tabla_fuente_split = pd.crosstab(
    df_split["dataset"],
    df_split["split"]
).reindex(
    columns=["train", "val", "test"],
    fill_value=0
)

display(tabla_fuente_split)

print("\nPorcentaje de cada fuente asignado a cada split:")

tabla_fuente_pct = (
    tabla_fuente_split
    .div(
        tabla_fuente_split.sum(axis=1),
        axis=0
    )
    * 100
)

display(tabla_fuente_pct.round(2))

split,train,val,test
dataset,,,
CCTV_YOLO,6,2,0
DCSASS,545,127,111
MNNIT,127,18,37
SINTETICO,8,2,2



Porcentaje de cada fuente asignado a cada split:


split,train,val,test
dataset,,,
CCTV_YOLO,75.00,25.00,0.00
DCSASS,69.60,16.22,14.18
MNNIT,69.78,9.89,20.33
SINTETICO,66.67,16.67,16.67


In [ ]:
# ------------------------------------------------------------
# Distribución conjunta de fuente, clase y split
# ------------------------------------------------------------

tabla_fuente_clase_split = pd.crosstab(
    index=df_split["dataset"],
    columns=[
        df_split["split"],
        df_split["clase_binaria"]
    ]
)

tabla_fuente_clase_split = (
    tabla_fuente_clase_split
    .rename(
        columns={
            0: "sin_ocultamiento",
            1: "ocultamiento"
        },
        level=1
    )
)

display(tabla_fuente_clase_split)

split                     test                         train               \
clase_binaria sin_ocultamiento ocultamiento sin_ocultamiento ocultamiento   
dataset                                                                     
CCTV_YOLO                    0            0                3            3   
DCSASS                     102            9              485           60   
MNNIT                       21           16               63           64   
SINTETICO                    1            1                0            8   

split                      val               
clase_binaria sin_ocultamiento ocultamiento  
dataset                                      
CCTV_YOLO                    1            1  
DCSASS                     112           15  
MNNIT                        6           12  
SINTETICO                    1            1

In [ ]:
# ------------------------------------------------------------
# 1. Ningún grupo experimental puede cruzar particiones
# ------------------------------------------------------------

splits_por_grupo = (
    df_split
    .groupby("grupo_experimental")["split"]
    .nunique()
)

assert splits_por_grupo.max() == 1

print(
    "Grupos experimentales presentes en múltiples splits:",
    (splits_por_grupo > 1).sum()
)


# ------------------------------------------------------------
# 2. Ningún video puede cruzar particiones
# ------------------------------------------------------------

splits_por_video = (
    df_split
    .groupby("grupo_video")["split"]
    .nunique()
)

assert splits_por_video.max() == 1

print(
    "Videos presentes en múltiples splits:",
    (splits_por_video > 1).sum()
)


# ------------------------------------------------------------
# 3. Ninguna persona conocida puede cruzar particiones
# ------------------------------------------------------------

personas_validas = df_split[
    df_split["persona_conocida"]
].copy()

personas_validas["grupo_persona"] = (
    personas_validas["dataset"].astype(str)
    + "__"
    + personas_validas["persona_id_limpio"]
)

splits_por_persona = (
    personas_validas
    .groupby("grupo_persona")["split"]
    .nunique()
)

assert splits_por_persona.max() == 1

print(
    "Personas conocidas presentes en múltiples splits:",
    (splits_por_persona > 1).sum()
)


# ------------------------------------------------------------
# 4. Las tres particiones deben contener ambas clases
# ------------------------------------------------------------

clases_por_split = (
    df_split
    .groupby("split")["clase_binaria"]
    .nunique()
)

assert (clases_por_split == 2).all()

print("\nClases diferentes por split:")
display(clases_por_split.to_frame("n_clases"))

Grupos experimentales presentes en múltiples splits: 0
Videos presentes en múltiples splits: 0
Personas conocidas presentes en múltiples splits: 0

Clases diferentes por split:


,n_clases
split,
test,2
train,2
val,2
